In [1]:
import pandas as pd
import re
import math
import matplotlib.pyplot as plt

dataset = [
    ("Hey, are we still on for lunch at 1?", "ham"),
    ("CONGRATULATIONS! You have won a $500 gift card. Reply WIN to claim now.", "spam"),
    ("Can you pick up some milk on your way home?", "ham"),
    ("URGENT: Your bank account has been locked. Click here to verify your identity.", "spam"),
    ("I'm running about 10 minutes late, start without me.", "ham"),
    ("Get a FREE custom website today! Limited time offer. Call 1-800-WEB-NOW.", "spam"),
    ("Did you see the game last night? That ending was crazy.", "ham"),
    ("Exclusive offer for our loyal customers: 50% off all luxury watches. Buy now!", "spam"),
    ("Don't forget to attach the report before you send that email.", "ham"),
    ("FINAL NOTICE: Your car warranty is about to expire. Press 1 to speak with an agent.", "spam"),
    ("Sounds good, see you tomorrow morning.", "ham"),
    ("Make $5000 a week working from home! No experience needed. Text MONEY to 8888.", "spam"),
    ("Happy birthday! Hope you have a great day.", "ham"),
    ("You have (1) new voicemail from a private number. Call 0900-LISTEN to retrieve.", "spam"),
    ("Are you free for a quick call this afternoon?", "ham"),
    ("LOAN APPROVED! You are eligible for up to $10,000. Click to deposit now.", "spam"),
    ("Let me know when you get there so I know you're safe.", "ham"),
    ("Win a free vacation to the Bahamas! Just text SUN to 555-5555.", "spam"),
    ("I'll send the draft over by tonight for your review.", "ham"),
    ("Your package could not be delivered. Please pay the $2.99 shipping fee here.", "spam")
]

# Now you can use this to build your DataFrame!

In [2]:

# 1. Clean and split into a list of dicts for the DataFrame
data = []
for sms, label in dataset:
    # Basic cleaning: lowercase and remove non-alphanumeric
    words = re.findall(r'\w+', sms.lower())
    for word in words:
        data.append({'word': word, 'label': label})
data[:5]

[{'word': 'hey', 'label': 'ham'},
 {'word': 'are', 'label': 'ham'},
 {'word': 'we', 'label': 'ham'},
 {'word': 'still', 'label': 'ham'},
 {'word': 'on', 'label': 'ham'}]

In [31]:
total_spam = 0
for sms , label in dataset:
    if label == 'spam':
        total_spam += 1

In [32]:
total_spam

10

In [3]:

# Define the data
data_dic = {
    'SMS': [],
    'Spam': [],
}

for sms in data:
    word = sms["word"]
    data_dic["SMS"].append(word)
    is_spam = 1 if sms["label"] == 'spam' else 0
    data_dic['Spam'].append(is_spam)
    


In [27]:

# Create the DataFrame
df = pd.DataFrame(data)
print(df.head())


    word label
0    hey   ham
1    are   ham
2     we   ham
3  still   ham
4     on   ham


In [29]:
len(df[df["label"] == 'spam'])

138

In [5]:
df[df['word'].isin(["are"])]

,word,label
1,are,ham
163,are,ham
175,are,spam


In [6]:
# Group, count occurrences, and rename the column cleanly
df = df.groupby(['word', 'label']).size().reset_index(name='word_count')




In [7]:
# Create a DataFrame containing only spam
df_spam = df[df['label'] == 'spam'].copy()

# Create a DataFrame containing only ham
df_ham = df[df['label'] == 'ham'].copy()

df


,word,label,word_count
0,000,spam,1
1,0900,spam,1
2,1,ham,1
3,1,spam,3
4,10,ham,1
...,...,...,...
176,working,spam,1
177,you,ham,8
178,you,spam,3
179,your,ham,2


In [8]:
df_ham[df_ham['word'].isin(['are'])]


,word,label,word_count
25,are,ham,2


In [9]:
total_spam = sum(df_spam['word_count'])
total_ham = sum(df_ham['word_count'])

In [10]:
total_words= total_ham + total_spam

In [ ]:
df[df["label"] == 'spam']

,word,label,word_count
0,000,spam,1
1,0900,spam,1
3,1,spam,3
5,10,spam,1
6,2,spam,1
...,...,...,...
173,with,spam,1
175,won,spam,1
176,working,spam,1
178,you,spam,3


In [12]:
# df_ham["probability"] = df_ham['word_count']/total_ham
# df_spam["probability"] = df_spam['word_count']/total_spam
df_ham = df_ham.drop(columns='label')
df_spam = df_spam.drop(columns='label')

In [13]:
df_spam[df_spam['word'] == 'you']

,word,word_count
178,you,3


In [14]:
def spam_prob(string:str):
    index = df_spam[df_spam['word'] == string]["word_count"]
    if not len(index) == 0: 
        prob = (index.values[0] + 1)/total_spam
    else:
        prob = 1 / total_spam
    return prob

def ham_prob(string:str):
    index = df_ham[df_ham["word"] == string]['word_count']
    if not len(index) == 0: 
        prob = (index.values[0] + 1)/total_spam
    else:
        prob = 1 / total_spam
    return float(prob)


In [15]:

ham_prob("you")
# index

0.06521739130434782

In [16]:
def is_spam(string :str):
    words = string.lower().strip().split()
    total_prob = 0
    for word in words:
        prob = spam_prob(word)
        log_prob = math.log(prob)
        total_prob += log_prob

    prob_is_spam = total_spam / total_words
    total_prob += prob_is_spam
    return total_prob

In [17]:
def is_ham(string :str):
    words = string.lower().strip().split()
    total_prob = 0
    for word in words:
        prob = ham_prob(word)
        log_prob = math.log(float(prob))
        total_prob += log_prob

    prob_is_ham = total_ham / total_words
    total_prob += prob_is_ham
    return total_prob

In [18]:
def clasifier(string :str):
    """
    Simple spam classifier
    Input : string , pass the message as an string
    Output :  [prob_spam , prob_ham] -> [probability of spam , probablility of ham]
    """
    prob_spam = is_spam(string)
    prob_ham = is_ham(string)

    if prob_spam > prob_ham:
        print("ALERT This is a SPAM")
    elif prob_ham > prob_spam:
        print("This message is safe")
    else:
        print("something went wrong Sorry!")

    return [prob_spam , prob_ham]

In [ ]:
class Classifier:
    """ 

    This is an simple Spam classifier \n
    Parameters to init are "dataset" : list (list of dictionary)\n
    Example :\n
    dataset = [\n
        ("Hey, are we still on for lunch at 1?", "ham"),\n
        ("CONGRATULATIONS! You have won a $500 gift card. Reply WIN to claim now.", "spam"),\n
        ]\n
    """
    def __init__(self , dataset : list):
        self.dataset = dataset
        self.init()

        self.df_spam : pd.DataFrame = None
        self.df_ham : pd.DataFrame = None
        self.total_spam = 0
        self.total_ham = len(dataset) - self.total_spam
        self.total_spam_word = 0
        self.total_ham_word = 0
        self.total =  self.total_spam + self.total_ham


    def init(self):
        data = []
        for sms, label in dataset:
            if label == 'spam':
                self.total_spam += 1
            # Basic cleaning: lowercase and remove non-alphanumeric
            words = re.findall(r'\w+', sms.lower())
            for word in words:
                data.append({'word': word, 'label': label})

        df = pd.DataFrame(data)


        df = df.groupby(['word', 'label']).size().reset_index(name='word_count')

        # Create a DataFrame containing only spam
        df_spam = df[df['label'] == 'spam'].copy()
        # Create a DataFrame containing only ham
        df_ham = df[df['label'] == 'ham'].copy()


        self.df_ham = df_ham.drop(columns='label')
        self.total_ham = sum(df_ham['word_count'])
        
        self.df_spam = df_spam.drop(columns='label')
        self.total_spam = sum(df_spam['word_count'])


    def _spam_prob(self , string:str):
        index = df_spam[df_spam['word'] == string]["word_count"]
        if not len(index) == 0: 
            prob = (index.values[0] + 1)/self.total_spam
        else:
            prob = 1 / self.total_spam
        return float(prob)

    def _ham_prob(self , string:str):
        index = df_ham[df_ham["word"] == string]['word_count']
        if not len(index) == 0: 
            prob = (index.values[0] + 1)/self.total_ham
        else:
            prob = 1 / self.total_ham
        return float(prob)

    def is_spam(self, string :str):
        words = string.lower().strip().split()
        total_prob = 0
        for word in words:
            prob = spam_prob(word)
            log_prob = math.log(prob)
            total_prob += log_prob

        prob_is_spam = self.total_spam / self.total
        total_prob += prob_is_spam
        return total_prob

    def is_ham(self, string :str):
        words = string.lower().strip().split()
        total_prob = 0
        for word in words:
            prob = ham_prob(word)
            log_prob = math.log(float(prob))
            total_prob += log_prob

        prob_is_ham = self.total_ham / self.total
        total_prob += prob_is_ham
        return total_prob

    def __call__(self,string :str):
        """
        Simple spam classifier
        Input : string , pass the message as an string
        Output : {
            'prob_spam' : prob_spam,
            'prob_ham' : prob_ham
        }
        """
        prob_spam = is_spam(string)
        prob_ham = is_ham(string)

        if prob_spam > prob_ham:
            print("ALERT This is a SPAM")
        elif prob_ham > prob_spam:
            print("This message is safe")
        else:
            print("something went wrong Sorry!")

        return {
            'prob_spam' : prob_spam,
            'prob_ham' : prob_ham
        }

        

In [39]:
clas = Classifier(dataset)

In [21]:
clas('you won Money')

ALERT This is a SPAM


{'prob_spam': -11.426893852219177, 'prob_ham': -12.166814959148052}

In [75]:

class Classifier:
    """ 

    This is an simple Spam classifier \n
    Parameters to init are "dataset" : list (list of dictionary)\n
    Example :\n
    dataset = [\n
        ("Hey, are we still on for lunch at 1?", "ham"),\n
        ("CONGRATULATIONS! You have won a $500 gift card. Reply WIN to claim now.", "spam"),\n
        ]\n
    """
    def __init__(self , dataset : list = None):
        self.dataset = dataset
        self.total_spam = 0
        self.total_spam_word = 0
        self.total_ham_word = 0

        self.spam_words = None
        self.ham_words = None
        self.vocab_size = None

        self.init()
        self.total_ham = len(dataset) - self.total_spam
        self.total_sms =  self.total_spam + self.total_ham

        self.total_word =  self.total_spam_word + self.total_ham_word


    def init(self):
        data = []
        for sms, label in dataset:
            if label == 'spam':
                self.total_spam += 1
            # Basic cleaning: lowercase and remove non-alphanumeric
            words = re.findall(r'\w+', sms.lower())
            for word in words:
                data.append({'word': word, 'label': label})

        df = pd.DataFrame(data)
        print(df.head())


        df = df.groupby(['word', 'label']).size().reset_index(name='word_count')
        self.vocab_size = len(set(df['word']))


        # Create a DataFrame containing only spam
        df_spam = df[df['label'] == 'spam'].copy()
        # Create a DataFrame containing only ham
        df_ham = df[df['label'] == 'ham'].copy()


        self.df_ham = df_ham.drop(columns='label')
        self.total_ham_word = sum(df_ham['word_count'])
        self.ham_words = dict(zip(df_ham['word'] ,df_ham['word_count']))
        
        self.df_spam = df_spam.drop(columns='label')
        self.total_spam_word = sum(df_spam['word_count'])
        self.spam_words = dict(zip(df_spam['word'] ,df_spam['word_count']))
        



    def _spam_prob(self , word:str):
        word_count = self.spam_words.get(word , 0)
        prob = (word_count + 1) / (self.total_spam_word + self.vocab_size)
        return float(prob)

    def _ham_prob(self , word:str):
        word_count = self.ham_words.get(word ,0)
        prob = (word_count + 1) / (self.total_ham_word + self.vocab_size)
        return float(prob)

    def is_spam(self, string :str):
        words = string.lower().strip().split()
        total_prob = 0
        for word in words:
            prob = self._spam_prob(word)
            log_prob = math.log(prob)
            total_prob += log_prob

        prob_is_spam = math.log(self.total_spam / self.total_sms)
        total_prob += prob_is_spam
        return total_prob

    def is_ham(self, string :str):
        words = string.lower().strip().split()
        total_prob = 0
        for word in words:
            prob = self._ham_prob(word)
            log_prob = math.log(float(prob))
            total_prob += log_prob

        prob_is_ham = math.log(self.total_ham / self.total_sms)
        total_prob += prob_is_ham
        return total_prob

    def __call__(self,string :str):
        """
        Simple spam classifier
        Input : string , pass the message as an string
        Output : {
            'prob_spam' : prob_spam,
            'prob_ham' : prob_ham
        }
        """
        prob_spam = self.is_spam(string)
        prob_ham = self.is_ham(string)

        if prob_spam > prob_ham:
            print("ALERT This is a SPAM")
        elif prob_ham > prob_spam:
            print("This message is safe")
        else:
            print("something went wrong Sorry!")

        return {
            'prob_spam' : prob_spam,
            'prob_ham' : prob_ham
        }

        

In [76]:
cls = Classifier(dataset)

    word label
0    hey   ham
1    are   ham
2     we   ham
3  still   ham
4     on   ham


In [84]:
cls("cup of coffe ")

This message is safe


{'prob_spam': -17.83434559708805, 'prob_ham': -17.420994489998893}